# ULTRON on Google Colab

Run the **full** Ultron app (dashboard, brain, tools, memory, SSE) on Colab's hardware instead of your own machine, reachable from your browser through a tunnel.

**What runs where**
- The FastAPI server, brain, tools, and dashboard run on the Colab VM.
- `run_shell` / `read_file` / `system_stats` act on the **Colab VM** (a throwaway Google container), not your computer.
- The LLM "brain" is still a cloud API (Groq here) — the same as running locally.

**Heads up: Colab sessions are temporary.** When the runtime resets, `ultron.db` and anything you didn't push to GitHub is gone. Cell 6 (optional) mounts Google Drive to persist the memory DB across sessions.

Run the cells top to bottom.

## 1. Clone (or update) the repo

In [ ]:
import os

REPO_URL = "https://github.com/Razoradams9/ultron.git"
REPO_DIR = "/content/ultron"

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

## 2. Install dependencies

The core app is lightweight. `pyngrok` is for the tunnel. Voice deps (Chatterbox/torch) are **not** installed here — do that in the optional voice cell only if you want spoken replies.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q pyngrok nest_asyncio

## 3. Set your Groq API key

Paste your key from [console.groq.com/keys](https://console.groq.com/keys). It lives only in this session's memory — it is not written to the repo.

Optional: uncomment to switch persona to the polite butler.

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Paste your GROQ_API_KEY: ").strip()
os.environ["ULTRON_PROVIDER"] = "groq"
# os.environ["ULTRON_PERSONA"] = "jarvis"   # sardonic 'ultron' is the default

print("Key set." if os.environ["GROQ_API_KEY"] else "No key entered — the brain won't respond.")

## 4. (Optional) Get an ngrok token

A free [ngrok](https://dashboard.ngrok.com/get-started/your-authtoken) authtoken gives you a stable tunnel to the dashboard. Skip this cell to use Colab's built-in port proxy instead (Cell 5 handles both).

In [ ]:
from getpass import getpass

token = getpass("Paste your ngrok authtoken (or press Enter to skip): ").strip()
if token:
    from pyngrok import ngrok
    ngrok.set_auth_token(token)
    print("ngrok token set.")
else:
    print("Skipping ngrok — will use Colab's port proxy.")

## 5. Launch the server + open the dashboard

Starts uvicorn in the background on port 8000, then exposes it. Click the printed URL to open the ULTRON control room.

Re-run this cell if you edit code and want a fresh server.

In [ ]:
import threading, time, uvicorn, nest_asyncio

nest_asyncio.apply()  # Colab already runs an event loop; let uvicorn share it

PORT = 8000

def _serve():
    uvicorn.run("ultron.server:app", host="0.0.0.0", port=PORT, log_level="warning")

threading.Thread(target=_serve, daemon=True).start()
time.sleep(4)  # let it bind

public_url = None
try:
    from pyngrok import ngrok
    public_url = ngrok.connect(PORT).public_url
    print("ULTRON dashboard:", public_url)
except Exception as e:
    print("ngrok unavailable (", e, ") — falling back to Colab proxy below.")
    try:
        from google.colab.output import eval_js
        print("ULTRON dashboard:", eval_js(f"google.colab.kernel.proxyPort({PORT})"))
    except Exception as e2:
        print("Colab proxy also unavailable:", e2)

## 6. (Optional) Persist memory across sessions

By default `ultron.db` is created in the repo folder and vanishes when the runtime resets. Run this cell **before Cell 5** to keep the conversation log + facts on your Google Drive instead.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.makedirs('/content/drive/MyDrive/ultron', exist_ok=True)
# # config.py reads ultron.db from the project root; symlink it to Drive:
# db = '/content/drive/MyDrive/ultron/ultron.db'
# link = '/content/ultron/ultron.db'
# if not os.path.islink(link):
#     if os.path.exists(link):
#         os.remove(link)
#     os.symlink(db, link)
# print('Memory persisted at', db)

## 7. (Optional) GPU-accelerated cloned voice

The voice service is the only heavy piece. Colab gives you a free GPU, so it runs far faster here than on CPU.

**First:** set the runtime to GPU — *Runtime → Change runtime type → T4 GPU*, then re-run cells 1–5.

**Upload your reference voice clip** (10-20s of clean single-speaker speech, no music) via the folder icon on the left before running this cell.

This cell installs Chatterbox **without letting it downgrade Colab's CUDA-enabled torch** (the `--no-deps` trick), registers your clip as the voiceprint, starts `voice/server.py` on the GPU, and points the main app at it. Re-run Cell 5 afterward so the app picks up `VOICE_URL`.

In [ ]:
import os, glob, threading, time

# Install Chatterbox WITHOUT --deps so it can't rip out Colab's CUDA torch,
# then add its real runtime deps separately.
!pip install -q --no-deps chatterbox-tts
!pip install -q pykakasi==2.3.0 pyloudnorm resemble-perth s3tokenizer spacy-pkuseg conformer
!pip install -q librosa soundfile

import torch
assert torch.cuda.is_available(), "CUDA not available — set Runtime > Change runtime type > T4 GPU and rerun cells 1-5."
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

os.environ["VOICE_DEVICE"] = "cuda"

# Register your uploaded clip as the voiceprint the service will speak in.
cands = []
for ext in ("*.mp3","*.wav","*.m4a","*.mpeg","*.ogg","*.flac"):
    cands += glob.glob("/content/"+ext)
cands = [c for c in cands if not c.endswith(("ref.wav","cloned.wav","ultron_test.wav"))]
cands.sort(key=os.path.getmtime, reverse=True)
if cands:
    import librosa, soundfile as sf
    os.makedirs("/content/ultron/voice/samples", exist_ok=True)
    y, sr = librosa.load(cands[0], sr=24000, mono=True)
    sf.write("/content/ultron/voice/samples/reference.wav", y, 24000, subtype="PCM_16")
    print("voiceprint registered from:", cands[0], f"({len(y)/sr:.0f}s)")
else:
    print("No clip found — service will use the stock voice. Upload a clip and rerun to clone.")

def _serve_voice():
    import uvicorn
    uvicorn.run("voice.server:app", host="127.0.0.1", port=8001, log_level="warning")

threading.Thread(target=_serve_voice, daemon=True).start()
time.sleep(4)

os.environ["VOICE_URL"] = "http://127.0.0.1:8001"
print("\nVoice service starting on :8001 (GPU). Now RE-RUN Cell 5 so the app uses VOICE_URL,")
print("then toggle VOICE: ON in the dashboard. First spoken reply loads the model (~30s), then it's fast.")